# Emotion story generation: Pilot

Generate stories with Qwen3-4B through Ollama. Start Ollama and run `ollama pull qwen3:4b`, then execute the cells in order.

This notebook generates a 56-story pilot: 14 emotions × 2 topics × 2 stories.

The prompt is adapted from [Sofroniew et al., Appendix 6.5](https://arxiv.org/abs/2604.07729), with one example slot per requested story. The labels include the twelve illustrated in Appendix 6.6 plus frustrated and hopeful; all topics come from Appendix 6.5.


In [1]:
import hashlib
import json
import re
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

MODEL = 'qwen3:4b'
URL = 'http://localhost:11434'

OUT = Path('data/pilot.jsonl')
USER_MESSAGE = 'Write the stories now. /no_think'
MAX_ATTEMPTS = 3
print('Output file:', OUT.resolve())


Output file: /home/p4sharma/private/dsc180a-testing/data/pilot.jsonl


In [2]:
EMOTIONS = ['desperate', 'nervous', 'surprised', 'calm', 'angry', 'loving',
            'sad', 'afraid', 'inspired', 'happy', 'guilty', 'proud',
            'frustrated', 'hopeful']

TOPICS = [
    'A neighbor starts a renovation project',
    'An adult child moves back in with their parents',
]

TERMS = {
    'happy': ['happy', 'happiness', 'joy', 'joyful', 'delighted', 'cheerful'],
    'calm': ['calm', 'calmness', 'serene', 'peaceful', 'relaxed'],
    'sad': ['sad', 'sadness', 'sorrow', 'sorrowful', 'unhappy'],
    'angry': ['angry', 'anger', 'furious', 'enraged', 'irate'],
    'frustrated': ['frustrated', 'frustration', 'exasperated'],
    'afraid': ['afraid', 'fear', 'fearful', 'scared', 'frightened', 'terrified'],
    'desperate': ['desperate', 'desperation'],
    'hopeful': ['hopeful', 'hope', 'optimistic', 'optimism'],
    'nervous': ['nervous', 'nervousness', 'anxious', 'anxiety', 'jittery'],
    'surprised': ['surprised', 'surprise', 'astonished', 'shocked', 'amazed'],
    'loving': ['loving', 'love', 'affection', 'adoration', 'devoted'],
    'inspired': ['inspired', 'inspiration', 'inspiring', 'motivated', 'invigorated'],
    'guilty': ['guilty', 'guilt', 'remorse', 'remorseful', 'ashamed'],
    'proud': ['proud', 'pride', 'prideful'],
}


In [3]:
def make_prompt(emotion, topic, count):

    slots = '\n'.join(f'<NEW STORY>\n[story {k}]' for k in range(1, count + 1))
    return f'''Write {count} different stories based on the following premise.
Topic: {topic}
The story should follow a character who is feeling {emotion}.
Format the stories like so:
{slots}
The paragraphs should each be a fresh start, with no continuity. Try to make them diverse and not use the same turns of phrase. Across the different stories, use a mix of third-person narration and first-person narration.
IMPORTANT: You must NEVER use the word '{emotion}' or any direct synonyms of it in the stories. Instead, convey the emotion ONLY through:
- The character's actions and behaviors
- Physical sensations and body language
- Dialogue and tone of voice
- Thoughts and internal reactions
- Situational context and environmental descriptions
The emotion should be clearly conveyed to the reader through these indirect means, but never explicitly named.'''

def check(story, emotion):

    flags = []
    if not 100 <= len(story.split()) <= 160:
        flags.append('outside_requested_word_range')
    hits = [term for term in TERMS[emotion]
            if re.search(r'\b' + re.escape(term) + r'\b', story, re.I)]
    if hits:
        flags.append('explicit_emotion_terms:' + ','.join(hits))
    return flags

MARKER_AT_START = re.compile(r'^\s*\[story\s*\d*\.?\]\s*', re.I)

def parse_stories(raw):

    notes = []
    text = raw
    if '</think>' in text:
        text = text.rsplit('</think>', 1)[1]
        notes.append('stripped_reasoning_before_think_end')
    elif '<think>' in text:
        notes.append('unclosed_think_block')
    parts = text.split('<NEW STORY>')
    if parts[0].strip():
        notes.append('text_before_first_new_story')
    stories = []
    for p in parts[1:]:
        cleaned = MARKER_AT_START.sub('', p).strip()
        if cleaned != p.strip() and 'removed_story_n_marker' not in notes:
            notes.append('removed_story_n_marker')
        if cleaned:
            stories.append(cleaned)
    return stories, notes

def reply_is_valid(stories, notes, count):

    return (len(stories) == count
            and 'text_before_first_new_story' not in notes
            and 'unclosed_think_block' not in notes)


## Run settings


In [4]:
topics = TOPICS
count = 2
config = dict(model=MODEL, url=URL.rstrip('/'), emotions=EMOTIONS,
              topics=topics, stories_per_cell=count, temperature=0.8,
              base_seed=180, prompt_version='3-paper-prompt-count-slots',
              user_message=USER_MESSAGE, max_attempts=MAX_ATTEMPTS,
              cleanup='strip-reasoning,strip-story-markers,strict-count',
              prompt_template=make_prompt('{emotion}', '{topic}', count))
run_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16]
print(f'{len(EMOTIONS)} emotions x {len(topics)} topics x {count} = '
      f'{len(EMOTIONS) * len(topics) * count} stories')
print('run_id:', run_id)

OUT.parent.mkdir(parents=True, exist_ok=True)
MANIFEST = OUT.with_suffix('.manifest.json')
RAW = OUT.with_suffix('.raw.jsonl')

if MANIFEST.exists():
    if json.loads(MANIFEST.read_text())['config'] != config:
        raise ValueError('Settings differ from existing manifest. Choose a new OUT path.')
    print('Existing manifest matches; will resume.')
elif OUT.exists():
    raise ValueError('Output exists without a manifest. Choose a new OUT path.')
else:
    MANIFEST.write_text(json.dumps(dict(run_id=run_id, config=config,
        created_utc=datetime.now(timezone.utc).isoformat()), indent=2))
    print('Wrote manifest:', MANIFEST)


14 emotions x 2 topics x 2 = 56 stories
run_id: df89db73de21bee8
Existing manifest matches; will resume.


## Preview the prompt


In [5]:
print(make_prompt(EMOTIONS[0], topics[0], count))


Write 2 different stories based on the following premise.
Topic: A neighbor starts a renovation project
The story should follow a character who is feeling desperate.
Format the stories like so:
<NEW STORY>
[story 1]
<NEW STORY>
[story 2]
The paragraphs should each be a fresh start, with no continuity. Try to make them diverse and not use the same turns of phrase. Across the different stories, use a mix of third-person narration and first-person narration.
IMPORTANT: You must NEVER use the word 'desperate' or any direct synonyms of it in the stories. Instead, convey the emotion ONLY through:
- The character's actions and behaviors
- Physical sensations and body language
- Dialogue and tone of voice
- Thoughts and internal reactions
- Situational context and environmental descriptions
The emotion should be clearly conveyed to the reader through these indirect means, but never explicitly named.


## Model details

In [6]:
def api_get(path):
    with urllib.request.urlopen(config['url'] + path, timeout=30) as r:
        return json.load(r)

print('Ollama version:', api_get('/api/version')['version'])
for m in api_get('/api/tags')['models']:
    if m['name'] == MODEL or m['model'] == MODEL:
        print('Model:', m['name'])
        print('Digest:', m['digest'])
        print('Details:', m.get('details'))
        break
else:
    print(f'Model {MODEL} not found. Run: ollama pull {MODEL}')


Ollama version: 0.40.15
Model: qwen3:4b
Digest: 359d7dd4bcdab3d86b87d73ac27966f4dbb9f5efdfcc75d34a8764a09474fae7
Details: {'parent_model': '', 'format': 'gguf', 'family': 'qwen3', 'families': ['qwen3'], 'parameter_size': '4.0B', 'quantization_level': 'Q4_K_M', 'context_length': 262144, 'embedding_length': 2560, 'runner': 'ggml'}


## Generate stories


In [7]:
rows = [json.loads(line) for line in OUT.read_text().splitlines()] if OUT.exists() else []
done = {r['id'] for r in rows}
texts = {r['text'].strip().casefold() for r in rows}

for ei, emotion in enumerate(EMOTIONS):
    for ti, topic in enumerate(topics):
        ids = [f'{run_id}-{ei:02d}-{ti:02d}-{i:02d}' for i in range(count)]
        if all(i in done for i in ids):
            continue
        prompt = make_prompt(emotion, topic, count)
        base_seed = config['base_seed'] + ei * len(topics) + ti
        for attempt in range(1, MAX_ATTEMPTS + 1):

            seed = base_seed + (attempt - 1) * 10000
            payload = dict(model=MODEL, stream=False, think=False,
                messages=[dict(role='system', content=prompt),
                          dict(role='user', content=USER_MESSAGE)],
                options=dict(temperature=config['temperature'], seed=seed, num_predict=2400))
            request = urllib.request.Request(config['url'] + '/api/chat',
                data=json.dumps(payload).encode(), headers={'Content-Type': 'application/json'})
            print(f'Generating {emotion} / topic {ti} (attempt {attempt}) ...', flush=True)
            with urllib.request.urlopen(request, timeout=600) as response:
                result = json.load(response)
            stories, notes = parse_stories(result['message']['content'])
            accepted = reply_is_valid(stories, notes, count)
            with RAW.open('a') as f:
                f.write(json.dumps(dict(run_id=run_id, emotion=emotion, topic=topic,
                    attempt=attempt, accepted=accepted, stories_found=len(stories),
                    parse_notes=notes, prompt=prompt, seed=seed, response=result),
                    ensure_ascii=False) + '\n')
            if accepted:
                break
            print(f'  rejected: found {len(stories)} stories, notes={notes}', flush=True)
        else:
            raise RuntimeError(f'No valid reply after {MAX_ATTEMPTS} attempts for {emotion} / topic {ti}. '
                               f'Inspect the replies in {RAW}.')
        with OUT.open('a') as f:
            for i, story in enumerate(stories):
                if ids[i] in done:
                    continue
                flags = check(story, emotion)
                if story.casefold() in texts:
                    flags.append('exact_duplicate')

                row = dict(id=ids[i], run_id=run_id, emotion=emotion, topic_id=ti,
                    topic=topic, story_index=i, split='pilot', text=story,
                    word_count=len(story.split()), flags=flags,
                    review_status='pending', model=result.get('model', MODEL), seed=seed,
                    attempt=attempt, cleanup=notes)
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
                f.flush()
                done.add(ids[i])
                texts.add(story.casefold())
        print(f'Saved {emotion} / topic {ti}: {count} stories (attempt {attempt})', flush=True)

print(f'Done: {OUT}. All stories still require human review.')


Generating desperate / topic 0 (attempt 1) ...
Saved desperate / topic 0: 2 stories (attempt 1)
Generating desperate / topic 1 (attempt 1) ...
Saved desperate / topic 1: 2 stories (attempt 1)
Generating nervous / topic 0 (attempt 1) ...
Saved nervous / topic 0: 2 stories (attempt 1)
Generating nervous / topic 1 (attempt 1) ...
Saved nervous / topic 1: 2 stories (attempt 1)
Generating surprised / topic 0 (attempt 1) ...
Saved surprised / topic 0: 2 stories (attempt 1)
Generating surprised / topic 1 (attempt 1) ...
Saved surprised / topic 1: 2 stories (attempt 1)
Generating calm / topic 0 (attempt 1) ...
Saved calm / topic 0: 2 stories (attempt 1)
Generating calm / topic 1 (attempt 1) ...
Saved calm / topic 1: 2 stories (attempt 1)
Generating angry / topic 0 (attempt 1) ...
Saved angry / topic 0: 2 stories (attempt 1)
Generating angry / topic 1 (attempt 1) ...
Saved angry / topic 1: 2 stories (attempt 1)
Generating loving / topic 0 (attempt 1) ...
Saved loving / topic 0: 2 stories (attem

## Inspect the results

The 100–160 word range is a screening check, not a prompt requirement.


In [8]:
rows = [json.loads(line) for line in OUT.read_text().splitlines()]
wc = [r['word_count'] for r in rows]
flagged = [r for r in rows if r['flags']]

print('Total stories:', len(rows))
print('Stories per emotion:', dict(Counter(r['emotion'] for r in rows)))
print('Stories per split:', dict(Counter(r['split'] for r in rows)))
print(f'Word count: min {min(wc)}, mean {sum(wc) / len(wc):.0f}, max {max(wc)}')
print(f'Flagged: {len(flagged)} / {len(rows)} ({len(flagged) / len(rows):.0%})')
print('Flag types:', dict(Counter(f.split(':')[0] for r in rows for f in r['flags'])))


Total stories: 56
Stories per emotion: {'desperate': 4, 'nervous': 4, 'surprised': 4, 'calm': 4, 'angry': 4, 'loving': 4, 'sad': 4, 'afraid': 4, 'inspired': 4, 'happy': 4, 'guilty': 4, 'proud': 4, 'frustrated': 4, 'hopeful': 4}
Stories per split: {'pilot': 56}
Word count: min 122, mean 161, max 224
Flagged: 27 / 56 (48%)
Flag types: {'outside_requested_word_range': 27}


In [12]:
SHOW_EMOTION = "hopeful"  # filter by emotion, use None if no filter
FLAGGED_ONLY = True

for story in rows:
    if SHOW_EMOTION and story['emotion'] != SHOW_EMOTION:
        continue
    if FLAGGED_ONLY and not story['flags']:
        continue
    print(f"[{story['id']}] {story['emotion']} | {story['word_count']} words")
    print(story['topic'])
    print('Flags:', story['flags'])
    print(story['text'])
    print('-' * 80)

[df89db73de21bee8-13-00-00] hopeful | 184 words
A neighbor starts a renovation project
Flags: ['outside_requested_word_range']
The hammer’s rhythm against the old shed door was the first sound Elara registered. She’d been sitting on the porch swing, her hands resting on her knees, watching the man in the yellow jumpsuit push a wheelbarrow loaded with soil down her street. His movements were deliberate, almost surgical. She didn’t move her own body, but her knuckles tightened slightly on the swing’s iron rail. The morning sun warmed the back of her neck, and for the first time in weeks, the familiar ache in her chest didn’t feel like stone but like a slow, steady pulse. She watched the dirt settle into the cracks of the sidewalk, the way the man’s shirt was stained with fresh paint, the way he paused to wipe sweat from his forehead before continuing. She thought of the garden she’d been trying to save, the way the dandelions had pushed through last year. *Maybe the garden might bloom ag

## Manual inspection

We will manually check a sample for the intended emotion and ensure it sticks to the provided topic, avoids explicit emotion words, and avoids repeated phrasing.
